In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark = SparkSession.builder.appName("Test data").getOrCreate()

In [2]:
from pyspark.sql.types import *

In [ ]:
gold = spark.read.parquet("D:/Work/Datahub/output/spark_lake/gold")

In [ ]:
#Layer 1 — Bronze: ingest CSV
# - Nạp CSV nguyên trạng.
# - Giữ được mọi bản ghi của người lao động theo `CREATED_AT`.
# - Chưa chọn dữ liệu mới nhất và chưa tổng hợp báo cáo.

In [ ]:
#Định nghĩa schema cho bảng bhxh
schema_bhxh = StructType([
    StructField("ID", LongType(), nullable=False),
    StructField("NLD_ID", LongType(), nullable=False),

    StructField("SO_SO_BHXH", StringType(), nullable=True),
    StructField("THANG_BD", StringType(), nullable=True),
    StructField("THANG_KT", StringType(), nullable=True),

    StructField("TT_TG_BHXH", StringType(), nullable=True),
    StructField("DT_TG_BHXH", StringType(), nullable=True),
    StructField("NAM_TG_BHXH", IntegerType(), nullable=True),
    StructField("THANG_TG_BHXH", IntegerType(), nullable=True),

    StructField("NAM_TG_BHXH_BB", IntegerType(), nullable=True),
    StructField("THANG_TG_BHXH_BB", IntegerType(), nullable=True),

    StructField("TT_TG_BHTN", StringType(), nullable=True),
    StructField("DT_TG_BHTN", StringType(), nullable=True),
    StructField("NAM_TG_BHTN", IntegerType(), nullable=True),
    StructField("THANG_TG_BHTN", IntegerType(), nullable=True),

    StructField("TT_TG_BHYT", StringType(), nullable=True),
    StructField("DT_TG_BHYT", StringType(), nullable=True),
    StructField("NAM_TG_BHYT", IntegerType(), nullable=True),
    StructField("THANG_TG_BHYT", IntegerType(), nullable=True),

    StructField("NAM_NO_BHXH", IntegerType(), nullable=True),
    StructField("THANG_NO_BHXH", IntegerType(), nullable=True),

    StructField("NAM_NO_BHTN", IntegerType(), nullable=True),
    StructField("THANG_NO_BHTN", IntegerType(), nullable=True),

    StructField("TT_TG_BH", StringType(), nullable=True),
    StructField("DT_TG_BH", StringType(), nullable=True),

    StructField("TU_THANG_DVI", StringType(), nullable=True),
    StructField("DEN_THANG_DVI", StringType(), nullable=True),
    StructField("DEN_THANG_HTTT", StringType(), nullable=True),
    StructField("DEN_THANG_BHTN", StringType(), nullable=True),

    StructField("THANG_BD_LT", StringType(), nullable=True),
    StructField("THANG_KT_LT", StringType(), nullable=True),
    StructField("SO_THANG_LT", IntegerType(), nullable=True),

    StructField("IS_ERRORS", IntegerType(), nullable=True),
    StructField("NGHI_VIEC", IntegerType(), nullable=True),
    StructField("IS_CONTINUE", IntegerType(), nullable=True),
    StructField("TRUY_DONG", IntegerType(), nullable=True),

    StructField("DEN_NGAY", StringType(), nullable=True),

    StructField("MA_CD", StringType(), nullable=True),
    StructField("MA_NHH", StringType(), nullable=True),
    StructField("DD_MA_DON_VI", StringType(), nullable=True),

    StructField("DD_THANG_DONG_DEN_XH", StringType(), nullable=True),
    StructField("DD_TY_LE_NO_BHXH", DecimalType(10, 4), nullable=True),

    StructField("DD_THANG_DONG_DEN_YT", StringType(), nullable=True),
    StructField("DD_TY_LE_NO_BHYT", DecimalType(10, 4), nullable=True),

    StructField("DD_THANG_DONG_DEN_TN", StringType(), nullable=True),
    StructField("DD_TY_LE_NO_BHTN", DecimalType(10, 4), nullable=True),

    StructField("DD_THANG_DONG_DEN_TNLD", StringType(), nullable=True),
    StructField("DD_TY_LE_NO_TNLD", DecimalType(10, 4), nullable=True),

    StructField("RAW_RESPONSE", StringType(), nullable=True),

    StructField(
        "CREATED_AT",
        TimestampType(),
        nullable=False,
    ),
])

#Đọc file RAW_QTTG_BHXH.csv
df_bronze_bhxh = spark.read.csv("D:/Work/Datahub/output/raw_qttg_1m/RAW_QTTG_BHXH.csv", header=True, schema=schema_bhxh)


In [29]:
#Show 20 dòng dữ liệu của bhxh
df_bronze_bhxh.show(20)

+---+-------+----------+--------+--------+--------------+----------+-----------+-------------+--------------+----------------+----------+----------+-----------+-------------+----------+----------+-----------+-------------+-----------+-------------+-----------+-------------+--------------+--------+------------+-------------+--------------+--------------+-----------+-----------+-----------+---------+---------+-----------+---------+--------+-----+------+------------+--------------------+----------------+--------------------+----------------+--------------------+----------------+----------------------+----------------+------------+-------------------+
| ID| NLD_ID|SO_SO_BHXH|THANG_BD|THANG_KT|    TT_TG_BHXH|DT_TG_BHXH|NAM_TG_BHXH|THANG_TG_BHXH|NAM_TG_BHXH_BB|THANG_TG_BHXH_BB|TT_TG_BHTN|DT_TG_BHTN|NAM_TG_BHTN|THANG_TG_BHTN|TT_TG_BHYT|DT_TG_BHYT|NAM_TG_BHYT|THANG_TG_BHYT|NAM_NO_BHXH|THANG_NO_BHXH|NAM_NO_BHTN|THANG_NO_BHTN|      TT_TG_BH|DT_TG_BH|TU_THANG_DVI|DEN_THANG_DVI|DEN_THANG_HTTT|DEN_

In [30]:
df_bronze_bhxh.printSchema()

root
 |-- ID: long (nullable = true)
 |-- NLD_ID: long (nullable = true)
 |-- SO_SO_BHXH: string (nullable = true)
 |-- THANG_BD: string (nullable = true)
 |-- THANG_KT: string (nullable = true)
 |-- TT_TG_BHXH: string (nullable = true)
 |-- DT_TG_BHXH: string (nullable = true)
 |-- NAM_TG_BHXH: integer (nullable = true)
 |-- THANG_TG_BHXH: integer (nullable = true)
 |-- NAM_TG_BHXH_BB: integer (nullable = true)
 |-- THANG_TG_BHXH_BB: integer (nullable = true)
 |-- TT_TG_BHTN: string (nullable = true)
 |-- DT_TG_BHTN: string (nullable = true)
 |-- NAM_TG_BHTN: integer (nullable = true)
 |-- THANG_TG_BHTN: integer (nullable = true)
 |-- TT_TG_BHYT: string (nullable = true)
 |-- DT_TG_BHYT: string (nullable = true)
 |-- NAM_TG_BHYT: integer (nullable = true)
 |-- THANG_TG_BHYT: integer (nullable = true)
 |-- NAM_NO_BHXH: integer (nullable = true)
 |-- THANG_NO_BHXH: integer (nullable = true)
 |-- NAM_NO_BHTN: integer (nullable = true)
 |-- THANG_NO_BHTN: integer (nullable = true)
 |-- TT

In [6]:
df_bhxh.count()

142857

In [ ]:
#Định nghĩa schema cho bảng bhxh_detail
schema__bhxh_detail = StructType([
    # Primary key / Foreign keys
    StructField("ID", LongType(), nullable=False),
    StructField("MASTER_ID", LongType(), nullable=False),
    StructField("NLD_ID", LongType(), nullable=False),

    # Period / Organization
    StructField("DOT_PHAT_SINH", StringType(), nullable=True),
    StructField("TU_THANG", StringType(), nullable=True),
    StructField("DEN_THANG", StringType(), nullable=True),
    StructField("MA_DON_VI", StringType(), nullable=True),
    StructField("TEN_DON_VI", StringType(), nullable=True),

    # Classification
    StructField("LOAI_DT", StringType(), nullable=True),
    StructField("LOAI", IntegerType(), nullable=True),
    StructField("PA", StringType(), nullable=True),
    StructField("DON_VI_TINH", StringType(), nullable=True),
    StructField("MA_NT", StringType(), nullable=True),

    # Position / Workplace
    StructField("CHUC_DANH_CV", StringType(), nullable=True),
    StructField("CHUC_DANH_CV_PRE", StringType(), nullable=True),
    StructField("NOI_LAM_VIEC", StringType(), nullable=True),
    StructField("NOI_DUNG", StringType(), nullable=True),

    # Salary
    StructField("MUC_LUONG", DecimalType(19, 4), nullable=True),
    StructField("MUC_LUONG_TN", DecimalType(19, 4), nullable=True),
    StructField("MUC_LUONG_BHYT", DecimalType(19, 4), nullable=True),
    StructField("MUC_LUONG_PC", DecimalType(19, 4), nullable=True),
    StructField("MUC_LUONG_BS", DecimalType(19, 4), nullable=True),
    StructField("MUC_LUONG_NLD", DecimalType(19, 4), nullable=True),
    StructField("MUC_LUONG_NSNN", DecimalType(19, 4), nullable=True),
    StructField("MUC_LUONG_HS", DecimalType(19, 4), nullable=True),
    StructField("MUC_LUONG_TT", DecimalType(19, 4), nullable=True),

    # Salary coefficients
    StructField("HS_LUONG", DecimalType(10, 4), nullable=True),
    StructField("PC_CHUC_VU", DecimalType(10, 4), nullable=True),
    StructField("PC_THAM_NIEN", DecimalType(10, 4), nullable=True),
    StructField("PC_NGHE", DecimalType(10, 4), nullable=True),
    StructField("PC_KHU_VUC", DecimalType(10, 4), nullable=True),
    StructField("PC_KHAC", DecimalType(10, 4), nullable=True),
    StructField("PC_TAI_CU", DecimalType(10, 4), nullable=True),
    StructField("HS_TN", DecimalType(10, 4), nullable=True),
    StructField("HS_NG", DecimalType(10, 4), nullable=True),
    StructField("HS_TC", DecimalType(10, 4), nullable=True),

    # Contribution rates
    StructField("TYLE_BHXH", DecimalType(10, 4), nullable=True),
    StructField("TYLE_BHYT", DecimalType(10, 4), nullable=True),
    StructField("TYLE_BHTN", DecimalType(10, 4), nullable=True),
    StructField("TYLE_TUDV", DecimalType(10, 4), nullable=True),
    StructField("TYLE_HTTT", DecimalType(10, 4), nullable=True),
    StructField("TYLE_ODTS", DecimalType(10, 4), nullable=True),
    StructField("TYLE_TNLD", DecimalType(10, 4), nullable=True),
    StructField("TYLE_NSNN", DecimalType(10, 4), nullable=True),

    # Conditions / flags
    StructField("DK1", IntegerType(), nullable=True),
    StructField("DK2", IntegerType(), nullable=True),
    StructField("DK3", IntegerType(), nullable=True),
    StructField("DK4", IntegerType(), nullable=True),
    StructField("DK5", IntegerType(), nullable=True),
    StructField("DK6", IntegerType(), nullable=True),

    # Insurance flags
    StructField("IS_BHXH", IntegerType(), nullable=True),
    StructField("IS_BHXH_BB", IntegerType(), nullable=True),
    StructField("IS_BHTN", IntegerType(), nullable=True),
    StructField("IS_BHYT", IntegerType(), nullable=True),
    StructField("IS_BHXH2", IntegerType(), nullable=True),
    StructField("IS_BHTN2", IntegerType(), nullable=True),

    # Status flags
    StructField("IS_ERROR", IntegerType(), nullable=True),
    StructField("IS_TR", IntegerType(), nullable=True),
    StructField("IS_BONUS", IntegerType(), nullable=True),
    StructField("ML_TC", IntegerType(), nullable=True),

    # Additional information
    StructField("GHI_CHU", StringType(), nullable=True),
    StructField("KIEM_TRA", IntegerType(), nullable=True),
    StructField("SO_THANG", IntegerType(), nullable=True),
    StructField("MA_KHOI_TK", StringType(), nullable=True),

    # Contribution
    StructField("TY_LE_DONG", DecimalType(10, 4), nullable=True),
    StructField("MUC_DONG", DecimalType(19, 4), nullable=True),

    # Main salary
    StructField("LUONG_CHINH", DecimalType(19, 4), nullable=True),

    # Position / contribution method
    StructField("CHUC_DANH_NLV", StringType(), nullable=True),
    StructField("PHUONG_THUC", StringType(), nullable=True),
    StructField("PHUONG_THUC_DONG", StringType(), nullable=True),

    # Previous salary information
    StructField("MUC_LUONG_PRE", DecimalType(19, 4), nullable=True),
    StructField("MUC_LUONG_PC_PRE", DecimalType(19, 4), nullable=True),
    StructField("MUC_LUONG_BS_PRE", DecimalType(19, 4), nullable=True),
    StructField("HS_LUONG_PRE", DecimalType(10, 4), nullable=True),
    StructField("PC_CHUC_VU_PRE", DecimalType(10, 4), nullable=True),
    StructField("PC_THAM_NIEN_PRE", DecimalType(10, 4), nullable=True),
    StructField("PC_NGHE_PRE", DecimalType(10, 4), nullable=True),
    StructField("PC_KHU_VUC_PRE", DecimalType(10, 4), nullable=True),
    StructField("PC_KHAC_PRE", DecimalType(10, 4), nullable=True),
    StructField("PC_TAI_CU_PRE", DecimalType(10, 4), nullable=True),
    StructField("LUONG_CHINH_PRE", DecimalType(19, 4), nullable=True),

    # Audit
    StructField("CREATED_AT", TimestampType(), nullable=False),
])

df_bronze_bhxh_detail = spark.read.csv("D:/Work/Datahub/output/raw_qttg_1m/RAW_QTTG_BHXH_DETAIL.csv", header=True,schema=schema_bhxh_detail)

In [55]:
df_bronze_bhxh_detail.show(20)

+---+---------+-------+-------------+--------+---------+---------+--------------------+-------+----+----+-----------+-----+--------------------+----------------+--------------+--------------------+-------------+------------+--------------+------------+------------+-------------+--------------+------------+------------+--------+----------+------------+-------+----------+-------+---------+-----+-----+-----+---------+---------+---------+---------+---------+---------+---------+---------+----+----+----+----+----+----+-------+----------+-------+-------+--------+--------+--------+-----+--------+-----+-------+--------+--------+----------+----------+--------+-----------+-------------+-----------+----------------+-------------+----------------+----------------+------------+--------------+----------------+-----------+--------------+-----------+-------------+---------------+-------------------+
| ID|MASTER_ID| NLD_ID|DOT_PHAT_SINH|TU_THANG|DEN_THANG|MA_DON_VI|          TEN_DON_VI|LOAI_DT|LOAI|  PA

In [47]:
df_bronze_bhxh_detail.printSchema()

root
 |-- ID: long (nullable = true)
 |-- MASTER_ID: long (nullable = true)
 |-- NLD_ID: long (nullable = true)
 |-- DOT_PHAT_SINH: string (nullable = true)
 |-- TU_THANG: string (nullable = true)
 |-- DEN_THANG: string (nullable = true)
 |-- MA_DON_VI: string (nullable = true)
 |-- TEN_DON_VI: string (nullable = true)
 |-- LOAI_DT: string (nullable = true)
 |-- LOAI: integer (nullable = true)
 |-- PA: string (nullable = true)
 |-- DON_VI_TINH: string (nullable = true)
 |-- MA_NT: string (nullable = true)
 |-- CHUC_DANH_CV: string (nullable = true)
 |-- CHUC_DANH_CV_PRE: string (nullable = true)
 |-- NOI_LAM_VIEC: string (nullable = true)
 |-- NOI_DUNG: string (nullable = true)
 |-- MUC_LUONG: decimal(19,4) (nullable = true)
 |-- MUC_LUONG_TN: decimal(19,4) (nullable = true)
 |-- MUC_LUONG_BHYT: decimal(19,4) (nullable = true)
 |-- MUC_LUONG_PC: decimal(19,4) (nullable = true)
 |-- MUC_LUONG_BS: decimal(19,4) (nullable = true)
 |-- MUC_LUONG_NLD: decimal(19,4) (nullable = true)
 |-- MU

In [ ]:
#Kiểm tra layer 1 - bronze
df_invalid_detail = df_bronze_bhxh_detail.join(
    df_bronze_bhxh, df_bronze_bhxh.ID == df_bronze_bhxh_detail.MASTER_ID, "left") \
    .filter(
        df_bronze_bhxh.ID.isNull() | 
        (df_bronze_bhxh.NLD_ID != df_bronze_bhxh_detail.NLD_ID)
    )

In [ ]:
df_invalid_month =df_bronze_bhxh_detail.filter(
    (~col("TU_THANG").rlike(r"^[0-9]{6}$")) |
    ( ~col("DEN_THANG").rlike(r"^[0-9]{6}$")) |
    (~substring(col("TU_THANG"), 5, 2).between("01", "12")) |
    (~substring(col("DEN_THANG"), 5, 2).between("01", "12")) |
    (col("TU_THANG") < col("DEN_THANG"))
)

In [69]:
#Kiểm tra dòng trong RAW_QTTG_BHXH
df_bronze_bhxh.count()

142857

In [70]:
#Kiểm tra dòng trong RAW_QTTG_BHXH_DETAIL
df_bronze_bhxh_detail.count()

1000000

In [72]:
#Kiểm tra bản ghi detail không hợp lệ
df_invalid_detail.count()

0

In [71]:
#Kiểm tra bản ghi có thông tin tháng không hợp lệ
df_invalid_month.count()

0

In [ ]:
#Layer 2 - Silver: 
#- Mỗi người chỉ còn một dòng mới nhất theo `CREATED_AT`.
# - Detail chỉ lấy từ dòng mới nhất đó.
# - Làm sạch kiểu tháng, số tiền và khóa liên kết.
# - Mỗi lần chạy sẽ soft-delete dữ liệu cũ.

In [84]:
from pyspark.sql.window import Window

#Dùng windown funtions row_number để nhóm cột "SO_SO_BHXH" vào và sắp xếp theo chiều giảm dần của cột "SO_SO_BHXH",
#nếu cột "SO_SO_BHXH" có giá trị giống nhau thì xét tiếp "ID" lấy cột theo "ID" mới nhất (cũng dùng desc để sắp xếp giảm dần) để có thể lấy được hàng mới nhất
window = Window.partitionBy("SO_SO_BHXH").orderBy(desc("CREATED_AT"),desc("ID"))
#Lấy hàng mới nhất có vị trí: rank = 1
df_latest_person_record = df_bronze_bhxh.withColumn("rank", row_number().over(window)).filter(col("rank") == 1)

#Xóa cột rank đi khi đã lấy được các cột mới nhất
df_silver_bhxh = df_latest_person_record.drop("rank")


In [92]:
#Lấy detail thuộc các master id vừa chọn

df_silver_bhxh_detail = df_latest_person_record.select("ID") \
    .join(df_bronze_bhxh_detail, df_latest_person_record.ID == df_bronze_bhxh_detail.MASTER_ID, "left").drop(df_latest_person_record.ID)


169658

In [95]:
#Kiểm tra Silver:

#Kiểm tra để biết mỗi người chỉ có 1 dòng Silver
df_silver_bhxh.select("SO_SO_BHXH").groupBy("SO_SO_BHXH").agg(count("*").alias("total")).where(col("total") != 1).show()

+----------+-----+
|SO_SO_BHXH|total|
+----------+-----+
+----------+-----+



In [103]:
#Kiểm tra Detail phải có dòng cha và đúng người
detail = df_silver_bhxh_detail.alias("d")
master = df_silver_bhxh.alias("m")
detail.join(master, master.ID == detail.MASTER_ID, "left") \
    .where(col("m.ID").isNull() | (col("m.NLD_ID") != col("d.NLD_ID"))).show()

+---+---------+------+-------------+--------+---------+---------+----------+-------+----+---+-----------+-----+------------+----------------+------------+--------+---------+------------+--------------+------------+------------+-------------+--------------+------------+------------+--------+----------+------------+-------+----------+-------+---------+-----+-----+-----+---------+---------+---------+---------+---------+---------+---------+---------+---+---+---+---+---+---+-------+----------+-------+-------+--------+--------+--------+-----+--------+-----+-------+--------+--------+----------+----------+--------+-----------+-------------+-----------+----------------+-------------+----------------+----------------+------------+--------------+----------------+-----------+--------------+-----------+-------------+---------------+----------+---+------+----------+--------+--------+----------+----------+-----------+-------------+--------------+----------------+----------+----------+-----------+----

In [ ]:
#Layer 3 — Gold: báo cáo tham gia BHXH theo tháng
# Tạo **Báo cáo tổng hợp tham gia BHXH theo tháng**.

#Cần trả lời:

# 1. Trong tháng có bao nhiêu người đang tham gia?
# 2. Có bao nhiêu đơn vị?
# 3. Tổng quỹ lương đóng là bao nhiêu?
# 4. Mức lương đóng bình quân là bao nhiêu?
# 5. Có bao nhiêu người có mức lương bằng 0?

#Cách làm: TU_THANG DEN_THANG
    #Vì cần từng tháng của các năm nên cần xử lý để lấy các tháng trong khoảng cột "TU_THANG" đến cột "DEN_THANG"
    # 2 cột đó đang là string nên phải chuyển về format về dạng date dùng to_date() (ví dụ 202609 thành 2026-09-01 )
    # sau đó tiếp tục tạo danh sách tháng "TU_THANG" - "COT_THANG",ví dụ: từ 1998-12-01  đến 1999-02-01 thành 1998-12-01,1999-01-01,1999-02-01 bằng hàm sequence
    # tiếp đến tách danh sách đó mỗi tháng thành một hàng (row) dùng hàm explose()
    #cuối cùng là chuyển về định dạng theo yêu cầu yyyyMM

In [141]:
df_test = df_silver_bhxh_detail\
    .withColumn("TU_THANG_DATE", to_date(col("TU_THANG"), 'yyyyMM'))\
    .withColumn("DEN_THANG_DATE", to_date(col("DEN_THANG"), 'yyyyMM'))\
    .withColumn("DS_THANG", sequence(col("TU_THANG_DATE"), col("DEN_THANG_DATE"), expr("INTERVAL 1 MONTH"))) \
    .withColumn("THANG", explode(col("DS_THANG"))) \
    .withColumn("THANG", date_format(col("THANG"), "yyyyMM")) \
    .select("NLD_ID","MA_DON_VI","TEN_DON_VI", "MUC_LUONG", "THANG").orderBy(desc(col("NLD_ID")), asc(col("THANG")))

In [144]:
df_test.count()

933644

In [142]:
df_result = df_test.groupBy("THANG").agg(
    count_distinct("NLD_ID").alias("SO_NGUOI_THAM_GIA"),
    count_distinct("MA_DON_VI").alias("SO_DON_VI_THAM_GIA"),
    sum("MUC_LUONG").alias("TONG_QUY_LUONG"),
    avg(
        when(col("MUC_LUONG") > 0, col("MUC_LUONG"))
    ).alias("LUONG_BINH_QUAN"),
    count_distinct(
        when(col("MUC_LUONG") == 0, col("NLD_ID"))
    ).alias("SO_NGUOI_CO_MUC_LUONG_0")
).orderBy(desc(col("THANG")))

In [143]:
df_result.show()

+------+-----------------+------------------+---------------+----------------+-----------------------+
| THANG|SO_NGUOI_THAM_GIA|SO_DON_VI_THAM_GIA| TONG_QUY_LUONG| LUONG_BINH_QUAN|SO_NGUOI_CO_MUC_LUONG_0|
+------+-----------------+------------------+---------------+----------------+-----------------------+
|202606|               41|                41| 184850000.0000|5436764.70588235|                      7|
|202605|               69|                69| 290450000.0000|5095614.03508772|                     12|
|202604|              109|               108| 473050000.0000|5500581.39534884|                     23|
|202603|              138|               136| 598550000.0000|5491284.40366972|                     29|
|202602|              179|               177| 836250000.0000|5575000.00000000|                     29|
|202601|              217|               213|1014100000.0000|5665363.12849162|                     38|
|202512|              251|               245|1179250000.0000|5752439.0243

In [139]:
df_result.show()

+------+-----------------+------------------+---------------+----------------+-----------------------+
| THANG|SO_NGUOI_THAM_GIA|SO_DON_VI_THAM_GIA| TONG_QUY_LUONG| LUONG_BINH_QUAN|SO_NGUOI_CO_MUC_LUONG_0|
+------+-----------------+------------------+---------------+----------------+-----------------------+
|202606|               41|                41| 184850000.0000|4508536.58536585|                      7|
|202605|               69|                69| 290450000.0000|4209420.28985507|                     12|
|202604|              109|               108| 473050000.0000|4339908.25688073|                     23|
|202603|              138|               136| 598550000.0000|4337318.84057971|                     29|
|202602|              179|               177| 836250000.0000|4671787.70949721|                     29|
|202601|              217|               213|1014100000.0000|4673271.88940092|                     38|
|202512|              251|               245|1179250000.0000|4698207.1713